# 🏎️ AI Mini Car Model Training

このノートブックでは、アノテーションデータを使用してステアリング・スロットル予測モデルを学習します。

## 利用可能なモデル
| モデル名 | 説明 | パラメータ数 |
|---------|------|-------------|
| donkeycar | Donkeycar標準モデル | ~100K |
| resnet18 | ResNet18ベース | ~11M |
| mobilevit_xxs | MobileViT超軽量 | ~1.3M |
| mobilenetv3_small_100 | MobileNetV3 Small | ~2.5M |
| mobilenetv4_conv_small | MobileNetV4 Small | ~3.8M |
| efficientnet_b0 | EfficientNet B0 | ~5.3M |
| efficientnetv2_s | EfficientNetV2 Small | ~21M |
| edgenext_xx_small | EdgeNeXt超軽量 | ~1.3M |
| efficientformer_l1 | EfficientFormer L1 | ~12M |

## 1. Google Driveのマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 必要なライブラリのインストール

In [ ]:
!pip install timm mlflow -q

## 3. 設定

以下のセルでデータパスとハイパーパラメータを設定してください。

In [ ]:
# ========================================
# データ設定（ここを変更してください）
# ========================================

# Google Drive上のフォルダ名
FOLDER_NAME = "annotation_data"

# ZIPファイル名（転送時に指定した名前）
DATA_FILE_NAME = "annotation_20241215_120000.zip"  # ← 実際のファイル名に変更

# データパス（自動設定）
DATA_PATH = f"/content/drive/MyDrive/{FOLDER_NAME}/{DATA_FILE_NAME}"
EXTRACT_PATH = "/content/data"

# ========================================
# モデル設定
# ========================================

# 使用するモデル（上記の表から選択）
MODEL_NAME = "resnet18"

# ========================================
# ハイパーパラメータ（ローカルと同じデフォルト値）
# ========================================

BATCH_SIZE = 32           # バッチサイズ
EPOCHS = 30               # エポック数（ローカルのデフォルト: 30）
LEARNING_RATE = 0.001     # 学習率
WEIGHT_DECAY = 1e-4       # 重み減衰（ローカルのデフォルト: 1e-4）
TRAIN_RATIO = 0.8         # 学習データの割合（検証: 0.2）

# 入力画像サイズ（自動取得 - 読み込んだ画像から設定されます）
# INPUT_SIZE は後で画像を読み込んだ際に自動設定されます

# Early Stopping設定
EARLY_STOPPING_PATIENCE = 5  # ローカルのデフォルト: 5

# 学習率スケジューラ（ローカルと同じReduceLROnPlateau）
USE_LR_SCHEDULER = True
LR_SCHEDULER_PATIENCE = 2    # ローカルのデフォルト: 2
LR_SCHEDULER_FACTOR = 0.5    # ローカルのデフォルト: 0.5

# ========================================
# MLflow設定
# ========================================

# MLflow実験名（ローカルと同じ名前）
MLFLOW_EXPERIMENT_NAME = "autonomous_driving"

# MLflowトラッキングを有効にする
MLFLOW_ENABLED = True

# 学習のコメント（任意）
TRAINING_COMMENT = ""

print(f"データパス: {DATA_PATH}")
print(f"モデル: {MODEL_NAME}")
print(f"ハイパーパラメータ:")
print(f"  - Epochs: {EPOCHS}")
print(f"  - Batch Size: {BATCH_SIZE}")
print(f"  - Learning Rate: {LEARNING_RATE}")
print(f"  - Weight Decay: {WEIGHT_DECAY}")
print(f"  - Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")
print(f"MLflow: {'有効' if MLFLOW_ENABLED else '無効'}")

## 3.5 MLflowのセットアップ

In [ ]:
import mlflow
import mlflow.pytorch
from datetime import datetime

if MLFLOW_ENABLED:
    # MLflowのトラッキングURIをGoogle Drive上に設定
    MLFLOW_TRACKING_DIR = f"/content/drive/MyDrive/{FOLDER_NAME}/mlruns"
    mlflow.set_tracking_uri(f"file://{MLFLOW_TRACKING_DIR}")
    
    # 実験を設定（ローカルと同じ名前）
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
    
    print(f"MLflow トラッキングURI: {MLFLOW_TRACKING_DIR}")
    print(f"実験名: {MLFLOW_EXPERIMENT_NAME}")
else:
    print("MLflowトラッキングは無効です")

## 4. データの展開

In [ ]:
import zipfile
import os
import sys

# ZIPファイルの存在確認
if not os.path.exists(DATA_PATH):
    print(f"エラー: データファイルが見つかりません: {DATA_PATH}")
    print(f"\nGoogle Drive/{FOLDER_NAME}/ にZIPファイルをアップロードしてください。")
else:
    # データを展開
    os.makedirs(EXTRACT_PATH, exist_ok=True)
    with zipfile.ZipFile(DATA_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    
    print(f"データを展開しました: {EXTRACT_PATH}")
    print(f"ファイル一覧: {os.listdir(EXTRACT_PATH)}")
    
    # imagesフォルダ内のファイル数を確認
    images_dir = os.path.join(EXTRACT_PATH, "images")
    if os.path.exists(images_dir):
        print(f"画像数: {len(os.listdir(images_dir))}")
    
    # パスを追加
    sys.path.insert(0, EXTRACT_PATH)
    
    # 共通モジュールの検出
    USE_LOCAL_MODEL_CATALOG = False
    USE_LOCAL_MODEL_TRAINING = False
    
    # model_catalog.py（モデル定義）
    if os.path.exists(os.path.join(EXTRACT_PATH, "model_catalog.py")):
        USE_LOCAL_MODEL_CATALOG = True
        print(f"✓ model_catalog.py を検出 - ローカルと同じモデル定義を使用します")
    else:
        print(f"✗ model_catalog.py が見つかりません - ノートブック内のモデル定義を使用します")
    
    # model_info.py（モデル情報）
    if os.path.exists(os.path.join(EXTRACT_PATH, "model_info.py")):
        print(f"✓ model_info.py を検出")
    else:
        print(f"✗ model_info.py が見つかりません")
    
    # model_training.py（学習ユーティリティ）
    if os.path.exists(os.path.join(EXTRACT_PATH, "model_training.py")):
        USE_LOCAL_MODEL_TRAINING = True
        print(f"✓ model_training.py を検出 - EarlyStopping等を使用します")
    else:
        print(f"✗ model_training.py が見つかりません - ノートブック内の実装を使用します")

## 5. データの読み込み

In [ ]:
import json

def load_annotations(data_path):
    """カタログファイルからアノテーションを読み込む"""
    annotations = []
    catalog_files = sorted([
        f for f in os.listdir(data_path)
        if f.endswith('.catalog') and not f.endswith('.catalog_manifest')
    ])

    for catalog_file in catalog_files:
        catalog_path = os.path.join(data_path, catalog_file)
        with open(catalog_path, 'r') as f:
            for line in f:
                if line.strip():
                    annotations.append(json.loads(line.strip()))

    return annotations

annotations = load_annotations(EXTRACT_PATH)
print(f"アノテーション数: {len(annotations)}")

# 最初のアノテーションの内容を確認
if annotations:
    print(f"\nアノテーション例:")
    for key, value in list(annotations[0].items())[:10]:
        print(f"  {key}: {value}")

## 6. PyTorch Dataset & DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

class DonkeyDataset(Dataset):
    """Donkeycar形式のデータセット（ローカルのAnnotationDatasetと前処理を統一）"""

    def __init__(self, annotations, images_dir, image_column='cam/image_array', transform=None, input_size=(120, 160)):
        self.annotations = annotations
        self.images_dir = images_dir
        self.image_column = image_column
        # ローカルと同じ前処理（Normalizeなし）
        self.transform = transform or transforms.Compose([
            transforms.Resize(input_size),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        record = self.annotations[idx]
        img_name = record[self.image_column]
        img_path = os.path.join(self.images_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        angle = record['user/angle']
        throttle = record['user/throttle']
        label = torch.tensor([angle, throttle], dtype=torch.float32)

        return image, label

In [ ]:
# DataLoaderの作成
images_dir = os.path.join(EXTRACT_PATH, "images")

# 画像カラムを検出
IMAGE_COLUMN = "cam/image_array"
if annotations:
    available_columns = [k for k in annotations[0].keys() if 'image_array' in k]
    if available_columns:
        IMAGE_COLUMN = available_columns[0]
        print(f"使用する画像カラム: {IMAGE_COLUMN}")

# 入力画像サイズを最初の画像から自動取得
first_img_name = annotations[0][IMAGE_COLUMN]
first_img_path = os.path.join(images_dir, first_img_name)
first_img = Image.open(first_img_path)
INPUT_SIZE = (first_img.height, first_img.width)  # (H, W)
print(f"入力画像サイズ（自動取得）: {INPUT_SIZE} (高さ, 幅)")

# データ分割
np.random.seed(42)
indices = np.random.permutation(len(annotations))
train_size = int(len(annotations) * TRAIN_RATIO)

train_annotations = [annotations[i] for i in indices[:train_size]]
val_annotations = [annotations[i] for i in indices[train_size:]]

print(f"学習データ数: {len(train_annotations)}")
print(f"検証データ数: {len(val_annotations)}")

# データセットとDataLoaderの作成
train_dataset = DonkeyDataset(train_annotations, images_dir, IMAGE_COLUMN, input_size=INPUT_SIZE)
val_dataset = DonkeyDataset(val_annotations, images_dir, IMAGE_COLUMN, input_size=INPUT_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## 7. モデルの定義

In [ ]:
import torch.nn as nn
import timm

# model_catalog.pyからget_modelを使用
if USE_LOCAL_MODEL_CATALOG:
    try:
        from model_catalog import get_model
        print("get_model を model_catalog.py からインポートしました")
        
        def create_model(model_name, input_size=(120, 160)):
            """model_catalog.pyのget_modelを使用してモデルを作成"""
            return get_model(model_name, pretrained=True, input_size=input_size, num_outputs=2)
    except ImportError as e:
        print(f"get_model のインポートに失敗: {e}")
        USE_LOCAL_MODEL_CATALOG = False

# model_training.pyからEarlyStoppingを使用
if USE_LOCAL_MODEL_TRAINING:
    try:
        from model_training import EarlyStopping, format_time, get_eta
        print("EarlyStopping, format_time, get_eta を model_training.py からインポートしました")
    except ImportError as e:
        print(f"model_training.py のインポートに失敗: {e}")
        USE_LOCAL_MODEL_TRAINING = False

# model_training.pyがない場合のフォールバック定義
if not USE_LOCAL_MODEL_TRAINING:
    print("ノートブック内のEarlyStopping定義を使用します")
    
    class EarlyStopping:
        """Early Stopping の実装"""
        def __init__(self, patience=5, min_delta=0, verbose=False):
            self.patience = patience
            self.min_delta = min_delta
            self.verbose = verbose
            self.counter = 0
            self.best_loss = None
            self.early_stop = False

        def __call__(self, val_loss):
            if self.best_loss is None:
                self.best_loss = val_loss
            elif val_loss < self.best_loss - self.min_delta:
                self.best_loss = val_loss
                self.counter = 0
            else:
                self.counter += 1
                if self.verbose:
                    print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
                if self.counter >= self.patience:
                    self.early_stop = True
                    return True
            return False
    
    def format_time(seconds):
        """秒数を時:分:秒の形式にフォーマット"""
        if seconds < 0:
            return "計算中..."
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        if hours > 0:
            return f"{hours}時間{minutes:02d}分{secs:02d}秒"
        elif minutes > 0:
            return f"{minutes}分{secs:02d}秒"
        else:
            return f"{secs}秒"

# model_catalog.pyがない場合のフォールバック定義
if not USE_LOCAL_MODEL_CATALOG:
    print("ノートブック内のモデル定義を使用します")
    
    class DonkeycarModel(nn.Module):
        """Donkeycar標準モデル（軽量CNN）"""
        def __init__(self, input_size=(120, 160)):
            super().__init__()
            self.conv = nn.Sequential(
                nn.Conv2d(3, 24, 5, stride=2), nn.ReLU(),
                nn.Conv2d(24, 32, 5, stride=2), nn.ReLU(),
                nn.Conv2d(32, 64, 5, stride=2), nn.ReLU(),
                nn.Conv2d(64, 64, 3, stride=2), nn.ReLU(),
                nn.Conv2d(64, 64, 3, stride=1), nn.ReLU(),
                nn.Flatten(),
            )
            with torch.no_grad():
                dummy = torch.zeros(1, 3, input_size[0], input_size[1])
                num_features = self.conv(dummy).shape[1]

            self.fc = nn.Sequential(
                nn.Linear(num_features, 100), nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(100, 50), nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(50, 2),
                nn.Tanh()
            )

        def forward(self, x):
            x = self.conv(x)
            return self.fc(x)

    class TimmModel(nn.Module):
        """TIMMライブラリを使用したモデル（ローカルと互換）"""
        def __init__(self, model_name, pretrained=True, num_outputs=2):
            super().__init__()
            self.base_model = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
            num_features = self.base_model.num_features
            self.regressor = nn.Linear(num_features, num_outputs)

        def forward(self, x):
            features = self.base_model(x)
            output = torch.tanh(self.regressor(features))
            return output

    def create_model(model_name, input_size=(120, 160)):
        """モデルを作成"""
        if model_name == "donkeycar":
            return DonkeycarModel(input_size=input_size)

        timm_model_map = {
            "resnet18": "resnet18",
            "mobilevit_xxs": "mobilevit_xxs",
            "mobilenetv3_small_100": "mobilenetv3_small_100",
            "mobilenetv4_conv_small": "mobilenetv4_conv_small.e2400_r224_in1k",
            "efficientnet_b0": "efficientnet_b0",
            "efficientnetv2_s": "efficientnetv2_rw_s",
            "edgenext_xx_small": "edgenext_xx_small",
            "efficientformer_l1": "efficientformer_l1",
        }

        if model_name in timm_model_map:
            return TimmModel(timm_model_map[model_name], pretrained=True)
        else:
            raise ValueError(f"Unknown model: {model_name}")

In [ ]:
# モデルの作成
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = create_model(MODEL_NAME, input_size=INPUT_SIZE).to(device)

print(f"デバイス: {device}")
print(f"モデル: {MODEL_NAME}")
print(f"パラメータ数: {sum(p.numel() for p in model.parameters()):,}")
print(f"学習可能パラメータ数: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 8. 学習

In [ ]:
from tqdm.notebook import tqdm
import time

criterion = nn.MSELoss()

# ローカルと同じ設定: Adam optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# ローカルと同じ設定: ReduceLROnPlateau scheduler
if USE_LR_SCHEDULER:
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min', 
        patience=LR_SCHEDULER_PATIENCE, 
        factor=LR_SCHEDULER_FACTOR
    )
    print(f"学習率スケジューラ: ReduceLROnPlateau (patience={LR_SCHEDULER_PATIENCE}, factor={LR_SCHEDULER_FACTOR})")

# Early Stopping
early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, verbose=True)

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'lr': []}
start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start_time = time.time()
    
    # 学習フェーズ
    model.train()
    train_loss = 0.0
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')

    for images, labels in train_pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    train_loss /= len(train_loader)

    # 検証フェーズ
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    # 学習率を記録
    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(current_lr)

    # エポック時間
    epoch_time = time.time() - epoch_start_time
    elapsed_time = time.time() - start_time
    
    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, "
          f"LR: {current_lr:.6f}, Time: {format_time(epoch_time)}, Elapsed: {format_time(elapsed_time)}")

    # 学習率スケジューラの更新（ReduceLROnPlateauは検証損失を監視）
    if USE_LR_SCHEDULER:
        scheduler.step(val_loss)

    # ベストモデルの保存
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'config': {
                'model_name': MODEL_NAME,
                'input_size': INPUT_SIZE,
                'output_size': 2
            }
        }, '/content/best_model.pt')
        print(f"  -> ベストモデルを保存しました (Val Loss: {best_val_loss:.4f})")

    # Early Stopping チェック
    if early_stopping(val_loss):
        print(f"\nEarly Stopping: {EARLY_STOPPING_PATIENCE}エポック改善がありませんでした")
        break

total_time = time.time() - start_time
print(f"\n学習完了! Best Val Loss: {best_val_loss:.4f}, Total Time: {format_time(total_time)}")

## 9. 結果の可視化

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 損失
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].set_title('Training & Validation Loss')
axes[0].grid(True)

# 学習率
axes[1].plot(history['lr'], label='Learning Rate', color='green', marker='^')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning Rate')
axes[1].legend()
axes[1].set_title('Learning Rate Schedule')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 10. 予測結果の可視化

In [ ]:
from matplotlib.patches import Circle

def draw_annotation_circles(ax, image_size, actual_angle, actual_throttle, pred_angle, pred_throttle):
    """
    アノテーションツールと同じスタイルで円を描画
    
    Args:
        ax: matplotlib axis
        image_size: (width, height) of the image
        actual_angle: 実際のangle値 (-1 to 1)
        actual_throttle: 実際のthrottle値 (-1 to 1)
        pred_angle: 予測angle値 (-1 to 1)
        pred_throttle: 予測throttle値 (-1 to 1)
    """
    img_width, img_height = image_size
    
    # 座標計算（アノテーションツールと同じ計算式）
    # angle: -1（左）→ 1（右）を 0 → width にマッピング
    # throttle: -1（下）→ 1（上）を height → 0 にマッピング（Y軸反転）
    
    # 実際の値（赤い円）
    actual_x = (actual_angle + 1) / 2 * img_width
    actual_y = (1 - (actual_throttle + 1) / 2) * img_height
    
    # 予測値（水色の円）
    pred_x = (pred_angle + 1) / 2 * img_width
    pred_y = (1 - (pred_throttle + 1) / 2) * img_height
    
    # 円のサイズ（画像サイズに応じてスケール）
    circle_radius = min(img_width, img_height) * 0.08
    line_width = 3
    
    # 赤い円：実際のアノテーション（塗りつぶしなし）
    actual_circle = Circle(
        (actual_x, actual_y), 
        circle_radius, 
        fill=False, 
        edgecolor='red', 
        linewidth=line_width,
        label='Actual'
    )
    ax.add_patch(actual_circle)
    
    # 水色の円：推論結果（塗りつぶしなし）
    pred_circle = Circle(
        (pred_x, pred_y), 
        circle_radius, 
        fill=False, 
        edgecolor='cyan', 
        linewidth=line_width,
        label='Predicted'
    )
    ax.add_patch(pred_circle)
    
    # 差分ベクトル（緑の矢印）- 実際 → 予測
    ax.annotate(
        '', 
        xy=(pred_x, pred_y), 
        xytext=(actual_x, actual_y),
        arrowprops=dict(arrowstyle='->', color='lime', lw=2)
    )

model.eval()

# サンプルを取得
sample_indices = np.random.choice(len(val_annotations), min(8, len(val_annotations)), replace=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

with torch.no_grad():
    for i, idx in enumerate(sample_indices):
        if i >= 8:
            break

        record = val_annotations[idx]
        img_name = record[IMAGE_COLUMN]
        img_path = os.path.join(images_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        img_width, img_height = image.size

        # 推論（Normalizeなし - ローカルと同じ前処理）
        transform = transforms.Compose([
            transforms.Resize(INPUT_SIZE),
            transforms.ToTensor()
        ])
        img_tensor = transform(image).unsqueeze(0).to(device)
        pred = model(img_tensor).cpu().numpy()[0]

        # 実際の値
        actual_angle = record['user/angle']
        actual_throttle = record['user/throttle']

        # 画像を表示
        axes[i].imshow(image)
        
        # 円を描画（アノテーションツールと同じスタイル）
        draw_annotation_circles(
            axes[i], 
            (img_width, img_height),
            actual_angle, actual_throttle,
            pred[0], pred[1]
        )
        
        # タイトル（値の表示）
        axes[i].set_title(
            f'Pred: A={pred[0]:.2f}, T={pred[1]:.2f}\n'
            f'Actual: A={actual_angle:.2f}, T={actual_throttle:.2f}',
            fontsize=9
        )
        axes[i].axis('off')

# 凡例を追加
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='none', 
           markeredgecolor='red', markersize=10, markeredgewidth=2, label='Actual (実際)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='none', 
           markeredgecolor='cyan', markersize=10, markeredgewidth=2, label='Predicted (予測)'),
    Line2D([0], [0], color='lime', lw=2, label='Difference (差分)')
]
fig.legend(handles=legend_elements, loc='upper center', ncol=3, fontsize=10, 
           bbox_to_anchor=(0.5, 0.02))

plt.tight_layout()
plt.subplots_adjust(bottom=0.08)
plt.show()

print("凡例: 赤=実際のアノテーション, 水色=推論結果, 緑矢印=差分ベクトル")

## 11. モデルの保存

In [ ]:
from datetime import datetime, timezone, timedelta

# 日本時間（JST = UTC+9）でタイムスタンプを生成
JST = timezone(timedelta(hours=9))
timestamp = datetime.now(JST).strftime("%Y%m%d_%H%M%S")

# アノテーションツールと同じ命名規則: {MODEL_NAME}_{timestamp}.pth
model_save_path = f"/content/drive/MyDrive/{FOLDER_NAME}/{MODEL_NAME}_{timestamp}.pth"

# ベストモデルを読み込んで保存
best_checkpoint = torch.load('/content/best_model.pt')
torch.save({
    'model_state_dict': best_checkpoint['model_state_dict'],
    'config': {
        'model_name': MODEL_NAME,
        'input_size': INPUT_SIZE,
        'output_size': 2,
        # 学習パラメータも保存（ローカルと同じ形式）
        'epochs': EPOCHS,
        'completed_epochs': len(history['train_loss']),
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'batch_size': BATCH_SIZE,
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
    },
    'val_loss': best_checkpoint['val_loss'],
    'training_history': history,
    'training_environment': 'colab',
}, model_save_path)

print(f"モデルを保存しました: {model_save_path}")
print(f"Best Validation Loss: {best_checkpoint['val_loss']:.4f}")
print(f"学習パラメータ:")
print(f"  - Epochs: {EPOCHS} (完了: {len(history['train_loss'])})")
print(f"  - Learning Rate: {LEARNING_RATE}")
print(f"  - Weight Decay: {WEIGHT_DECAY}")
print(f"  - Optimizer: Adam")
print(f"  - LR Scheduler: ReduceLROnPlateau")

## 11.5 MLflowへの記録

学習結果をMLflowに記録します。ローカルのアノテーションツールと同じ形式で保存されるため、後でダウンロードしてマージできます。

In [ ]:
if MLFLOW_ENABLED:
    # 実行名を生成（アノテーションツールと同じ形式）
    run_name = f"autonomous_driving_{MODEL_NAME}_{len(annotations)}samples_{timestamp}"
    
    with mlflow.start_run(run_name=run_name) as run:
        # ========================================
        # パラメータの記録（ローカルと同じ形式）
        # ========================================
        params = {
            # 基本パラメータ
            "framework": "pytorch",
            "model_type": MODEL_NAME,
            "data_folder": FOLDER_NAME,
            "task_type": "autonomous_driving",
            
            # 学習パラメータ（ローカルと同じ形式）
            "epochs": EPOCHS,
            "completed_epochs": len(history['train_loss']),
            "learning_rate": LEARNING_RATE,
            "batch_size": BATCH_SIZE,
            "optimizer": "Adam",           # ローカルと同じ
            "weight_decay": WEIGHT_DECAY,  # ローカルと同じ
            
            # 学習率スケジューラ
            "lr_scheduler": "ReduceLROnPlateau" if USE_LR_SCHEDULER else "None",
            "lr_scheduler_patience": LR_SCHEDULER_PATIENCE if USE_LR_SCHEDULER else 0,
            "lr_scheduler_factor": LR_SCHEDULER_FACTOR if USE_LR_SCHEDULER else 0,
            
            # Early Stopping
            "early_stopping": "enabled",
            "patience": EARLY_STOPPING_PATIENCE,
            
            # 入力サイズ
            "input_height": INPUT_SIZE[0],
            "input_width": INPUT_SIZE[1],
            
            # データセット情報
            "total_samples": len(annotations),
            "train_samples": len(train_annotations),
            "val_samples": len(val_annotations),
            "train_ratio": TRAIN_RATIO,
        }
        
        # コメントがあれば追加
        if TRAINING_COMMENT:
            params["comment"] = TRAINING_COMMENT
        
        mlflow.log_params(params)
        
        # ========================================
        # メトリクスの記録
        # ========================================
        metrics = {
            "best_val_loss": best_val_loss,
            "final_train_loss": history['train_loss'][-1] if history['train_loss'] else 0.0,
            "final_val_loss": history['val_loss'][-1] if history['val_loss'] else 0.0,
        }
        mlflow.log_metrics(metrics)
        
        # エポックごとのメトリクスも記録
        for epoch_idx, (train_l, val_l) in enumerate(zip(history['train_loss'], history['val_loss'])):
            mlflow.log_metric("train_loss_epoch", train_l, step=epoch_idx)
            mlflow.log_metric("val_loss_epoch", val_l, step=epoch_idx)
        
        # ========================================
        # タグの記録
        # ========================================
        tags = {
            "model_category": "autonomous_driving",
            "task_type": "regression",
            "framework": "pytorch",
            "status": "completed",
            "training_environment": "colab",  # Colabで学習したことを記録
        }
        mlflow.set_tags(tags)
        
        # ========================================
        # アーティファクト（モデル）の記録
        # ========================================
        # モデルファイルをアーティファクトとして保存
        mlflow.log_artifact(model_save_path, artifact_path="model")
        
        print(f"\nMLflowに記録しました:")
        print(f"  実行ID: {run.info.run_id}")
        print(f"  実行名: {run_name}")
        print(f"  トラッキングURI: {MLFLOW_TRACKING_DIR}")
        print(f"\n記録内容:")
        print(f"  パラメータ: {len(params)}項目")
        print(f"  メトリクス: best_val_loss={best_val_loss:.4f}")
        print(f"  タグ: training_environment=colab")
else:
    print("MLflowトラッキングは無効のためスキップしました")

## 12. ONNX形式でエクスポート（オプション）

推論用にONNX形式でエクスポートする場合は以下のセルを実行してください。

In [ ]:
# ONNXエクスポート
import torch.onnx

onnx_save_path = model_save_path.replace('.pth', '.onnx')

# ベストモデルをロード
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

# ダミー入力
dummy_input = torch.randn(1, 3, INPUT_SIZE[0], INPUT_SIZE[1]).to(device)

# ONNXにエクスポート
torch.onnx.export(
    model,
    dummy_input,
    onnx_save_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"ONNXモデルを保存しました: {onnx_save_path}")